# RAGSec Guard - eval recovery (tuned model only)

The full training run (v2) finished training and printed **tuned accuracy
0.9224 vs base 0.5366**, but its last cell crashed while saving results
(a JSON bug), so the tuned per-class scores were lost. The trained adapter
survived in v2's output.

This notebook re-runs **only the tuned evaluation** (~15-25 min):
attach TWO inputs - the `ragsec-guard-data` dataset and the previous
notebook's output (which contains `ragsec-guard-adapter`) - then
Save & Run All. Output: a complete `eval_results.json`.

In [ ]:
# 0. Check the GPU you got
!nvidia-smi

In [ ]:
# 1. Install Unsloth + training stack (Kaggle recipe)
!pip install -q unsloth
!pip install -q --no-deps trl peft accelerate bitsandbytes

In [ ]:
# 2. Locate the test file and the saved adapter (both attached as inputs)
import os

def find_file(name):
    for root, _dirs, files in os.walk("/kaggle/input"):
        if name in files:
            return os.path.join(root, name)
    raise FileNotFoundError(f"{name} not found - attach the dataset / notebook output first.")

TEST_PATH = find_file("guard_test.jsonl")
ADAPTER_DIR = os.path.dirname(find_file("adapter_config.json"))
print("test:   ", TEST_PATH)
print("adapter:", ADAPTER_DIR)

In [ ]:
# 3. Config + the base-model scores recovered from the v2 run
BASE_MODEL = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
MAX_SEQ_LEN = 1024
EVAL_LIMIT = 0          # 0 = full 915-row test set

BASE_SCORES = {
  "accuracy": 0.5366,
  "per_class": {
    "safe": {
      "precision": 0.524,
      "recall": 0.939,
      "f1": 0.672
    },
    "injection": {
      "precision": 0.333,
      "recall": 0.015,
      "f1": 0.029
    },
    "jailbreak": {
      "precision": 0.575,
      "recall": 0.69,
      "f1": 0.627
    }
  },
  "false_positive_rate_on_safe": 0.0606,
  "confusion": {}
}

In [ ]:
# 4. Load the base model (4-bit) and attach the trained adapter
import torch
from unsloth import FastLanguageModel
from peft import PeftModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)
model = PeftModel.from_pretrained(model, ADAPTER_DIR)
print("tuned model ready")

In [ ]:
# 5. Evaluation helpers - identical to the training notebook,
#    with the JSON bug fixed (confusion keys are strings now)
import gc
import json
import re
from collections import Counter

LABELS = ["safe", "injection", "jailbreak"]

def load_rows(path):
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]

test_rows = load_rows(TEST_PATH)
if EVAL_LIMIT:
    test_rows = test_rows[:EVAL_LIMIT]
print(f"evaluating on {len(test_rows)} held-out rows")

def build_prompt(row):
    messages = [{"role": "user",
                 "content": row["instruction"] + "\n\n" + row["input"]}]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True)

def parse_label(text):
    first = text.strip().splitlines()[0].strip().lower() if text.strip() else ""
    for lab in LABELS:
        if first.startswith(lab):
            return lab
    m = re.search(r"\b(safe|injection|jailbreak)\b", text.lower())
    return m.group(1) if m else "unparsed"

@torch.no_grad()
def run_eval(mdl, batch_size=16):
    FastLanguageModel.for_inference(mdl)
    preds = []
    prompts = [build_prompt(r) for r in test_rows]
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i + batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True,
                           truncation=True, max_length=MAX_SEQ_LEN).to("cuda")
        out = mdl.generate(**inputs, max_new_tokens=48, do_sample=False,
                           pad_token_id=tokenizer.pad_token_id)
        for j in range(len(batch)):
            gen = out[j][inputs["input_ids"].shape[1]:]
            preds.append(parse_label(tokenizer.decode(gen, skip_special_tokens=True)))
        if (i // batch_size) % 10 == 0:
            print(f"  {i + len(batch)}/{len(prompts)}")
    return preds

def score(preds):
    gold = [r["label"] for r in test_rows]
    acc = sum(p == g for p, g in zip(preds, gold)) / len(gold)
    per = {}
    for lab in LABELS:
        tp = sum(p == lab and g == lab for p, g in zip(preds, gold))
        fp = sum(p == lab and g != lab for p, g in zip(preds, gold))
        fn = sum(p != lab and g == lab for p, g in zip(preds, gold))
        prec = tp / (tp + fp) if tp + fp else 0.0
        rec = tp / (tp + fn) if tp + fn else 0.0
        per[lab] = {"precision": round(prec, 3), "recall": round(rec, 3),
                    "f1": round(2 * prec * rec / (prec + rec), 3) if prec + rec else 0.0}
    safe_fp = (sum(p != "safe" and g == "safe" for p, g in zip(preds, gold))
               / max(1, sum(g == "safe" for g in gold)))
    return {"accuracy": round(acc, 4), "per_class": per,
            "false_positive_rate_on_safe": round(safe_fp, 4),
            "confusion": {f"{g} -> {p}": n for (g, p), n in
                          Counter((g, p) for g, p in zip(gold, preds)).items()}}

In [ ]:
# 6. Evaluate the TUNED model
tuned_preds = run_eval(model)
tuned_scores = score(tuned_preds)
print("tuned:", tuned_scores["accuracy"])

In [ ]:
# 7. The table that matters + save everything
results = {"base": BASE_SCORES, "tuned": tuned_scores,
           "test_rows": len(test_rows),
           "base_model": BASE_MODEL,
           "note": "base scores from the v2 full run; tuned scores recovered "
                   "by this eval-only rerun against the v2 adapter"}
with open("/kaggle/working/eval_results.json", "w") as f:
    json.dump(results, f, indent=2)

base_scores = BASE_SCORES
print(f"{'metric':<28}{'base':>10}{'tuned':>10}")
print(f"{'accuracy':<28}{base_scores['accuracy']:>10}{tuned_scores['accuracy']:>10}")
for lab in LABELS:
    print(f"{'F1 ' + lab:<28}{base_scores['per_class'][lab]['f1']:>10}"
          f"{tuned_scores['per_class'][lab]['f1']:>10}")
print(f"{'false alarms on safe':<28}"
      f"{base_scores['false_positive_rate_on_safe']:>10}"
      f"{tuned_scores['false_positive_rate_on_safe']:>10}")
print("\nsaved: results -> /kaggle/working/eval_results.json")

## After it finishes

Download `eval_results.json` from this notebook's Output tab and send it
back - that completes the base-vs-tuned table for the writeup and resume.